# Member Cliams Prd Gold Metric Views - Setup Notebook
This notebook creates all metric views required for the **HealthPlan Metric Views KPI Dashboard**.
- **Target Schema**: `aira_test.aibi_workshop_schema`
- **Source Tables**: `dim_member`, `fact_claim_detail`, `fact_member_enrollment`
- **Metric Views Created**: 6 (mv_membership, mv_claims, mv_member_cost_profile, mv_member_lifecycle, mv_utilization, mv_new_enrollment)

> **Note**: Run cells sequentially. Each CREATE OR REPLACE VIEW is idempotent and safe to re-run.

## 1. mv_membership
Membership KPIs sourced from `dim_member`. Tracks total/active members, subscribers, PCP assignment, Medicaid counts, and deceased members.

In [0]:
CREATE OR REPLACE VIEW aira_test.aibi_workshop_schema.mv_membership
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "Membership KPIs - tracks member demographics, activity status, and PCP assignment rates"
source: aira_test.aibi_workshop_schema.dim_member
dimensions:
  - name: Line of Business
    expr: mbr_line_of_business
  - name: Line of Business Name
    expr: mbr_line_of_business_name
  - name: State
    expr: mbr_state
  - name: Sex
    expr: mbr_sex
  - name: Relationship Type
    expr: mbr_relationship_type
measures:
  - name: Total Members
    expr: COUNT(1)
  - name: Active Members
    expr: COUNT(1) FILTER (WHERE is_active = true)
  - name: Active Subscribers
    expr: COUNT(1) FILTER (WHERE mbr_sub_flag = '1' AND is_active = true)
  - name: Members with PCP
    expr: COUNT(1) FILTER (WHERE mbr_current_pcp_nbr IS NOT NULL AND is_active = true)
  - name: PCP Assigned Rate
    expr: MEASURE(`Members with PCP`) / MEASURE(`Active Members`)
  - name: Medicaid Member Count
    expr: COUNT(1) FILTER (WHERE mbr_line_of_business = 'MCD')
  - name: Deceased Member Count
    expr: COUNT(1) FILTER (WHERE mbr_deceased_flag = 'Y')
$$;

result


## 2. mv_claims
Claims KPIs sourced from `fact_claim_detail` joined with `dim_member` for Line of Business. Includes rolling window measures for 7-day claims, running totals, and 90-day clean claim rates.

In [0]:
CREATE OR REPLACE VIEW aira_test.aibi_workshop_schema.mv_claims
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "Claims KPIs - tracks claim volumes, paid/allowed/billed amounts, clean claim rates, and rolling window metrics"
source: >
  SELECT c.*, m.mbr_line_of_business, m.mbr_line_of_business_name
  FROM aira_test.aibi_workshop_schema.fact_claim_detail c
  LEFT JOIN aira_test.aibi_workshop_schema.dim_member m
  ON c.clm_dtl_member_nbr_sk = m.mbr_source_member_id
dimensions:
  - name: Service Date
    expr: clm_dtl_specific_dos_date
  - name: Service Month
    expr: DATE_TRUNC('MONTH', clm_dtl_specific_dos_date)
  - name: Service Year
    expr: CAST(YEAR(clm_dtl_specific_dos_date) AS STRING)
  - name: Benefit Category
    expr: clm_dtl_benefit_category
  - name: Claim Type
    expr: >
      CASE
        WHEN clm_dtl_place_of_service = '21' THEN 'Inpatient'
        WHEN clm_dtl_place_of_service IN ('22', '23') THEN 'Outpatient'
        WHEN clm_dtl_place_of_service = '11' THEN 'Professional'
        WHEN clm_dtl_benefit_category = 'Pharmacy' THEN 'Pharmacy'
        ELSE 'Other'
      END
  - name: Line of Business
    expr: mbr_line_of_business
measures:
  - name: Total Claims
    expr: COUNT(DISTINCT clm_dtl_claim_id)
  - name: Total Claim Lines
    expr: COUNT(1)
  - name: Total Paid Amount
    expr: SUM(clm_dtl_paid_amt)
  - name: Total Allowed Amount
    expr: SUM(clm_dtl_allowed_amt)
  - name: Total Billed Amount
    expr: SUM(clm_dtl_billed_amt)
  - name: Avg Paid per Claim
    expr: MEASURE(`Total Paid Amount`) / MEASURE(`Total Claims`)
  - name: Clean Claim Count
    expr: COUNT(1) FILTER (WHERE clm_dtl_clean_claim_ind = 'Y')
  - name: Clean Claim Rate
    expr: MEASURE(`Clean Claim Count`) / MEASURE(`Total Claim Lines`)
  - name: Inpatient Claim Count
    expr: COUNT(DISTINCT clm_dtl_claim_id) FILTER (WHERE clm_dtl_place_of_service = '21')
  - name: Rolling 7d Claims
    expr: COUNT(DISTINCT clm_dtl_claim_id)
    window:
      - order: Service Date
        range: trailing 7 day
        semiadditive: last
  - name: Running Total Paid
    expr: SUM(clm_dtl_paid_amt)
    window:
      - order: Service Month
        range: cumulative
        semiadditive: last
  - name: Rolling 90d Clean Count
    expr: COUNT(1) FILTER (WHERE clm_dtl_clean_claim_ind = 'Y')
    window:
      - order: Service Date
        range: trailing 90 day
        semiadditive: last
  - name: Rolling 90d Total Count
    expr: COUNT(1)
    window:
      - order: Service Date
        range: trailing 90 day
        semiadditive: last
  - name: Rolling 90d Clean Rate
    expr: MEASURE(`Rolling 90d Clean Count`) / MEASURE(`Rolling 90d Total Count`)
$$;

result


## 3. mv_member_cost_profile
Member cost analysis sourced from `dim_member` joined with aggregated claims. Tracks per-member costs, high-cost member identification, and PCP utilization.

In [0]:
CREATE OR REPLACE VIEW aira_test.aibi_workshop_schema.mv_member_cost_profile
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "Member Cost Profile - per-member cost analysis with high-cost identification and PCP utilization"
source: >
  SELECT
    m.member_sk,
    m.mbr_line_of_business,
    m.mbr_line_of_business_name,
    m.mbr_state,
    m.mbr_sex,
    m.is_active as member_is_active,
    COALESCE(c.total_paid, 0) as member_total_paid,
    COALESCE(c.claim_count, 0) as member_claim_count,
    CASE WHEN COALESCE(c.total_paid, 0) > 50000 THEN 1 ELSE 0 END as is_high_cost_member,
    CASE WHEN COALESCE(c.pcp_visit_count, 0) > 0 THEN 1 ELSE 0 END as has_pcp_visit
  FROM aira_test.aibi_workshop_schema.dim_member m
  LEFT JOIN (
    SELECT
      clm_dtl_member_nbr_sk,
      SUM(clm_dtl_paid_amt) as total_paid,
      COUNT(DISTINCT clm_dtl_claim_id) as claim_count,
      SUM(CASE WHEN clm_dtl_place_of_service = '11' THEN 1 ELSE 0 END) as pcp_visit_count
    FROM aira_test.aibi_workshop_schema.fact_claim_detail
    GROUP BY clm_dtl_member_nbr_sk
  ) c ON m.mbr_source_member_id = c.clm_dtl_member_nbr_sk
dimensions:
  - name: Line of Business Name
    expr: mbr_line_of_business_name
  - name: Line of Business
    expr: mbr_line_of_business
  - name: State
    expr: mbr_state
  - name: Sex
    expr: mbr_sex
measures:
  - name: Active Members
    expr: COUNT(1) FILTER (WHERE member_is_active = true)
  - name: Members with Claims
    expr: COUNT(1) FILTER (WHERE member_claim_count > 0)
  - name: Total Paid Amount
    expr: SUM(member_total_paid)
  - name: Avg Cost per Member
    expr: MEASURE(`Total Paid Amount`) / MEASURE(`Active Members`)
  - name: High-Cost Member Count
    expr: SUM(is_high_cost_member)
  - name: High-Cost Member Spend Pct
    expr: SUM(member_total_paid) FILTER (WHERE is_high_cost_member = 1) / MEASURE(`Total Paid Amount`)
  - name: PCP Utilization Rate
    expr: CAST(SUM(has_pcp_visit) FILTER (WHERE member_is_active = true) AS DOUBLE) / MEASURE(`Active Members`)
$$;

result


## 4. mv_member_lifecycle
Member lifecycle tracking - first enrollment dates, rolling new enrollment counts, and cumulative YTD enrollments.

In [0]:
CREATE OR REPLACE VIEW aira_test.aibi_workshop_schema.mv_member_lifecycle
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "Member Lifecycle - tracks first enrollment dates with rolling and cumulative enrollment metrics"
source: >
  SELECT
    e.member_sk,
    m.mbr_line_of_business,
    m.mbr_state,
    e.first_enrollment_date
  FROM (
    SELECT
      member_sk,
      MIN(mbr_enr_effective_date) as first_enrollment_date
    FROM aira_test.aibi_workshop_schema.fact_member_enrollment
    GROUP BY member_sk
  ) e
  JOIN aira_test.aibi_workshop_schema.dim_member m ON e.member_sk = m.member_sk
dimensions:
  - name: First Enrollment Date
    expr: first_enrollment_date
  - name: First Enrollment Month
    expr: DATE_TRUNC('MONTH', first_enrollment_date)
  - name: First Enrollment Year
    expr: DATE_TRUNC('YEAR', first_enrollment_date)
  - name: Line of Business
    expr: mbr_line_of_business
  - name: State
    expr: mbr_state
measures:
  - name: New Enrollments
    expr: COUNT(1)
  - name: Rolling 30d New Enrollments
    expr: COUNT(1)
    window:
      - order: First Enrollment Date
        range: trailing 30 day
        semiadditive: last
  - name: Cumulative YTD New Enrollments
    expr: COUNT(1)
    window:
      - order: First Enrollment Month
        range: cumulative
        semiadditive: last
      - order: First Enrollment Year
        range: current
        semiadditive: last
$$;

result


## 5. mv_utilization
Utilization and PMPM metrics at the member-month grain. Combines enrollment member months with claims data for per-member-per-month cost analysis, claims rates, and rolling/YTD trends.

In [0]:
CREATE OR REPLACE VIEW aira_test.aibi_workshop_schema.mv_utilization
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "Utilization & PMPM - member-month grain combining enrollment with claims for cost and utilization analysis"
source: >
  WITH member_months AS (
    SELECT DISTINCT
      e.member_sk,
      m.mbr_source_member_id,
      m.mbr_line_of_business,
      DATE_TRUNC('MONTH', e.mbr_enr_effective_date) as enrollment_month
    FROM aira_test.aibi_workshop_schema.fact_member_enrollment e
    JOIN aira_test.aibi_workshop_schema.dim_member m ON e.member_sk = m.member_sk
  ),
  monthly_claims AS (
    SELECT
      clm_dtl_member_nbr_sk,
      DATE_TRUNC('MONTH', clm_dtl_specific_dos_date) as service_month,
      SUM(clm_dtl_paid_amt) as paid_amount,
      COUNT(DISTINCT clm_dtl_claim_id) as claim_count
    FROM aira_test.aibi_workshop_schema.fact_claim_detail
    GROUP BY clm_dtl_member_nbr_sk, DATE_TRUNC('MONTH', clm_dtl_specific_dos_date)
  )
  SELECT
    mm.member_sk,
    mm.mbr_line_of_business,
    mm.enrollment_month,
    COALESCE(mc.paid_amount, 0) as monthly_paid,
    COALESCE(mc.claim_count, 0) as monthly_claims,
    CASE WHEN COALESCE(mc.claim_count, 0) > 0 THEN 1 ELSE 0 END as has_claims_flag
  FROM member_months mm
  LEFT JOIN monthly_claims mc
    ON mm.mbr_source_member_id = mc.clm_dtl_member_nbr_sk
    AND mm.enrollment_month = mc.service_month
dimensions:
  - name: Enrollment Month
    expr: enrollment_month
  - name: Enrollment Year
    expr: DATE_TRUNC('YEAR', enrollment_month)
  - name: Line of Business
    expr: mbr_line_of_business
measures:
  - name: Member Months
    expr: COUNT(1)
  - name: Total Paid
    expr: SUM(monthly_paid)
  - name: Total Claims
    expr: SUM(monthly_claims)
  - name: PMPM
    expr: MEASURE(`Total Paid`) / MEASURE(`Member Months`)
  - name: Claims per 1000 Members
    expr: MEASURE(`Total Claims`) / MEASURE(`Member Months`) * 1000
  - name: Utilization Rate
    expr: CAST(SUM(has_claims_flag) AS DOUBLE) / MEASURE(`Member Months`)
  - name: Rolling 3mo Paid
    expr: SUM(monthly_paid)
    window:
      - order: Enrollment Month
        range: trailing 3 month
        semiadditive: last
  - name: Rolling 3mo Member Months
    expr: COUNT(1)
    window:
      - order: Enrollment Month
        range: trailing 3 month
        semiadditive: last
  - name: Rolling 3mo PMPM
    expr: MEASURE(`Rolling 3mo Paid`) / MEASURE(`Rolling 3mo Member Months`)
  - name: YTD Paid
    expr: SUM(monthly_paid)
    window:
      - order: Enrollment Month
        range: cumulative
        semiadditive: last
      - order: Enrollment Year
        range: current
        semiadditive: last
  - name: YTD Member Months
    expr: COUNT(1)
    window:
      - order: Enrollment Month
        range: cumulative
        semiadditive: last
      - order: Enrollment Year
        range: current
        semiadditive: last
  - name: YTD PMPM
    expr: MEASURE(`YTD Paid`) / MEASURE(`YTD Member Months`)
  - name: Current Month Members
    expr: COUNT(DISTINCT member_sk)
    window:
      - order: Enrollment Month
        range: current
        semiadditive: last
  - name: Previous Month Members
    expr: COUNT(DISTINCT member_sk)
    window:
      - order: Enrollment Month
        range: trailing 1 month
        semiadditive: last
  - name: MoM Active Member Growth Pct
    expr: (MEASURE(`Current Month Members`) - MEASURE(`Previous Month Members`)) / MEASURE(`Previous Month Members`) * 100
$$;

result


## 6. mv_new_enrollment
New enrollment tracking for filter support and enrollment analysis. Provides Line of Business, State, and enrollment month dimensions.

In [0]:
CREATE OR REPLACE VIEW aira_test.aibi_workshop_schema.mv_new_enrollment
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "New Enrollment - tracks first enrollment events for filter support and enrollment trend analysis"
source: >
  SELECT
    e.member_sk,
    m.mbr_line_of_business,
    m.mbr_state,
    e.first_enrollment_date
  FROM (
    SELECT
      member_sk,
      MIN(mbr_enr_effective_date) as first_enrollment_date
    FROM aira_test.aibi_workshop_schema.fact_member_enrollment
    GROUP BY member_sk
  ) e
  JOIN aira_test.aibi_workshop_schema.dim_member m ON e.member_sk = m.member_sk
dimensions:
  - name: Line of Business
    expr: mbr_line_of_business
  - name: State
    expr: mbr_state
  - name: First Enrollment Date
    expr: first_enrollment_date
  - name: First Enrollment Month
    expr: DATE_TRUNC('MONTH', first_enrollment_date)
measures:
  - name: New Enrollments
    expr: COUNT(1)
$$;

result


## Verification Queries
Run these queries to verify the metric views were created successfully and return valid data.

In [0]:
-- Verify mv_membership
SELECT `Line of Business Name`, MEASURE(`Total Members`), MEASURE(`Active Members`), MEASURE(`PCP Assigned Rate`)
FROM aira_test.aibi_workshop_schema.mv_membership
GROUP BY ALL;

Line of Business Name,MEASURE(Total Members),MEASURE(Active Members),MEASURE(PCP Assigned Rate)
Medicare,8900,8707,1.0
Commercial,6681,6537,1.0
Exchange,975,962,1.0
Medicare-Medicaid Plan,493,485,1.0
Medicaid,2951,2877,1.0


In [0]:
-- Verify mv_claims
SELECT `Service Month`, MEASURE(`Total Claims`), MEASURE(`Total Paid Amount`), MEASURE(`Clean Claim Rate`)
FROM aira_test.aibi_workshop_schema.mv_claims
GROUP BY ALL
ORDER BY 1 DESC
LIMIT 12;

Service Month,MEASURE(Total Claims),MEASURE(Total Paid Amount),MEASURE(Clean Claim Rate)
2026-03-01T00:00:00.000Z,7946,25549186.1763,0.7973201352003862
2026-02-01T00:00:00.000Z,7429,23999486.4046,0.7980085348506402
2026-01-01T00:00:00.000Z,8283,25323086.5995,0.7973568281938326
2025-12-01T00:00:00.000Z,8065,26035825.9015,0.7980975029726516
2025-11-01T00:00:00.000Z,7966,24689568.2335,0.795645898484484
2025-10-01T00:00:00.000Z,8296,26101050.3809,0.8111961057023644
2025-09-01T00:00:00.000Z,7801,24900625.8277,0.8014498095589139
2025-08-01T00:00:00.000Z,8180,26446048.1917,0.8037877896718033
2025-07-01T00:00:00.000Z,8222,26678769.8123,0.7933836331979106
2025-06-01T00:00:00.000Z,8000,25577774.9865,0.7925027033521567


In [0]:
-- Verify mv_member_cost_profile
SELECT `Line of Business Name`, MEASURE(`Active Members`), MEASURE(`Avg Cost per Member`), MEASURE(`High-Cost Member Count`), MEASURE(`High-Cost Member Spend Pct`)
FROM aira_test.aibi_workshop_schema.mv_member_cost_profile
GROUP BY ALL;

Line of Business Name,MEASURE(Active Members),MEASURE(Avg Cost per Member),MEASURE(High-Cost Member Count),MEASURE(High-Cost Member Spend Pct)
Commercial,6537,46747.878462,2507,0.580240
Medicare,8707,46813.701800,3386,0.585610
Exchange,962,45740.049032,350,0.559465
Medicare-Medicaid Plan,485,45907.435262,193,0.592437
Medicaid,2877,47253.864345,1150,0.593028


In [0]:
-- Verify mv_utilization
SELECT `Enrollment Month`, MEASURE(`PMPM`), MEASURE(`Claims per 1000 Members`), MEASURE(`Utilization Rate`)
FROM aira_test.aibi_workshop_schema.mv_utilization
GROUP BY ALL
ORDER BY 1 DESC
LIMIT 12;

Enrollment Month,MEASURE(PMPM),MEASURE(Claims per 1000 Members),MEASURE(Utilization Rate)
2025-12-01T00:00:00.000Z,6228.283563,687.5,0.5
2025-11-01T00:00:00.000Z,1097.896901,389.64241676942044,0.3205918618988903
2025-10-01T00:00:00.000Z,1079.729264,425.5079006772009,0.3306997742663657
2025-09-01T00:00:00.000Z,1068.389495,384.1743119266055,0.3211009174311927
2025-08-01T00:00:00.000Z,1323.045843,406.3564131668558,0.3348467650397276
2025-07-01T00:00:00.000Z,1203.116189,426.85851318944844,0.34292565947242204
2025-06-01T00:00:00.000Z,1344.761960,394.5409429280397,0.3163771712158809
2025-05-01T00:00:00.000Z,1307.763496,431.9526627218935,0.35384615384615387
2025-04-01T00:00:00.000Z,1401.434827,410.22443890274315,0.33541147132169574
2025-03-01T00:00:00.000Z,1109.227090,405.72792362768496,0.3198090692124105


In [0]:
-- Verify mv_member_lifecycle
SELECT `First Enrollment Month`, MEASURE(`New Enrollments`), MEASURE(`Rolling 30d New Enrollments`)
FROM aira_test.aibi_workshop_schema.mv_member_lifecycle
GROUP BY ALL
ORDER BY 1 DESC
LIMIT 12;

First Enrollment Month,MEASURE(New Enrollments),MEASURE(Rolling 30d New Enrollments)
2025-12-01T00:00:00.000Z,3,0
2025-11-01T00:00:00.000Z,312,307
2025-10-01T00:00:00.000Z,336,314
2025-09-01T00:00:00.000Z,365,356
2025-08-01T00:00:00.000Z,353,340
2025-07-01T00:00:00.000Z,391,382
2025-06-01T00:00:00.000Z,411,404
2025-05-01T00:00:00.000Z,413,390
2025-04-01T00:00:00.000Z,416,405
2025-03-01T00:00:00.000Z,456,443


In [0]:
-- Verify mv_new_enrollment
SELECT `Line of Business`, `State`, MEASURE(`New Enrollments`)
FROM aira_test.aibi_workshop_schema.mv_new_enrollment
GROUP BY ALL
ORDER BY 3 DESC
LIMIT 20;

Line of Business,State,MEASURE(New Enrollments)
MCR,TX,222
MCR,VA,219
MCR,WI,211
MCR,TN,207
MCR,MN,205
MCR,OK,205
MCR,MA,204
MCR,NC,203
MCR,AZ,203
MCR,OR,203
